# TCC — Pipeline completo no Google Colab

Notebook único para preparação dos dados e treinamento. A arquitetura experimental é **modular**: primeiro treinamos um **Modelo Geral** em dataset amplo; depois cada **LoRA Especialista** é treinada separadamente sobre o mesmo checkpoint geral usando seu próprio dataset pequeno. O Full Fine-Tune é um controle separado, usando o mesmo checkpoint geral e o mesmo dataset pequeno da LoRA correspondente.

**Não treinamos as LoRAs juntas.** `det_lora_01`, `det_lora_02`, etc. são adapters independentes. O artefato principal de uma LoRA é o diretório `lora_adapter/`, que depois é carregado sobre o `best.pt` do Modelo Geral.

YOLO26n é usado inicialmente para smoke tests e para validar o protocolo na T4.


## 1. Verificar GPU sem importar PyTorch/Ultralytics


In [ ]:
import os, sys, subprocess
print('Python:', sys.version.split()[0])
print('Interpretador:', sys.executable)
subprocess.run(['nvidia-smi'], check=False)


## 2. Montar Google Drive


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
drive.mount('/content/drive')
RUNTIME = Path('/content/drive/MyDrive/TCC_Code_Test_runtime')
for name in ['data', 'workspace', 'exports', 'weights']:
    (RUNTIME / name).mkdir(parents=True, exist_ok=True)
print('Persistência:', RUNTIME)


## 3. Clonar ou atualizar o repositório privado


In [ ]:
REPO = Path('/content/TCC_Code_Test')
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('GITHUB_TOKEN não encontrado em Colab > Secrets.')
netrc = Path.home() / '.netrc'
netrc.write_text(f'machine github.com\nlogin x-access-token\npassword {token}\n', encoding='utf-8')
netrc.chmod(0o600)
try:
    if REPO.exists():
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', 'https://github.com/ferreis/TCC_Code_Test.git', str(REPO)], check=True)
finally:
    netrc.unlink(missing_ok=True)
os.chdir(REPO)
print('Commit TCC:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())


## 4. Instalar e validar YOLO-Master


In [ ]:
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, 'scripts/setup_yolo_master.py', '--dir', '/content/YOLO-Master'], check=True)
check_code = r'''
import importlib.util, pathlib, ultralytics
print('Ultralytics/YOLO-Master:', ultralytics.__version__)
print('Caminho:', pathlib.Path(ultralytics.__file__).resolve())
ok = importlib.util.find_spec('ultralytics.utils.lora') is not None
print('LoRA disponível:', ok)
if not ok: raise SystemExit('ERRO: ultralytics.utils.lora não encontrado')
'''
subprocess.run([sys.executable, '-c', check_code], check=True)
print('AMBIENTE YOLO VALIDADO.')


## 5. Validar GPU e backend dentro do kernel


In [ ]:
import importlib, importlib.util, pathlib, torch, ultralytics
importlib.invalidate_caches()
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU:', p.name)
    print('VRAM:', round(p.total_memory / 1024**3, 2), 'GB')
print('YOLO-Master:', ultralytics.__version__)
print('LoRA disponível:', importlib.util.find_spec('ultralytics.utils.lora') is not None)


## 6. Conectar data/workspace/exports/weights ao Drive


In [ ]:
import shutil
for name in ['data', 'workspace', 'exports', 'weights']:
    target = RUNTIME / name
    link = REPO / name
    if link.is_symlink(): link.unlink()
    elif link.exists():
        if link.is_dir():
            for p in link.iterdir():
                dst = target / p.name
                if not dst.exists(): shutil.copytree(p, dst) if p.is_dir() else shutil.copy2(p, dst)
            shutil.rmtree(link)
        else: link.unlink()
    link.symlink_to(target, target_is_directory=True)
print('Drive conectado ao projeto.')


## 7. Roboflow — dataset amplo de detecção


In [ ]:
rf_key = userdata.get('ROBOFLOW_API_KEY')
if not rf_key: raise RuntimeError('ROBOFLOW_API_KEY não encontrado em Secrets.')
os.environ['ROBOFLOW_API_KEY'] = rf_key
os.environ['ROBOFLOW_WORKSPACE'] = 'rafael-fernando-dos-reis-mecabo'
os.environ['ROBOFLOW_PROJECT'] = 'detecao-de-cetaceos-afrzs-iqey9'
os.environ['ROBOFLOW_VERSION'] = '1'
images_dir = Path('data/roboflow/train/images')
if images_dir.exists() and any(images_dir.iterdir()):
    print('Dataset já existe:', images_dir)
else:
    subprocess.run([sys.executable, 'scripts/download_roboflow.py', '--format', 'yolo26', '--output', 'data/roboflow', '--overwrite'], check=True)


## 8. SAM e clustering/revisão — opcionais


In [ ]:
sam = Path('weights/sam_vit_b_01ec64.pth')
if not sam.exists(): subprocess.run([sys.executable, 'scripts/download_sam.py'], check=True)
print('SAM:', sam)


In [ ]:
from tcc_discovery.box_clustering import BoxClusteringConfig, run_box_clustering
project_dir = Path('workspace/bbox_clusters_colab')
if (project_dir / 'catalog.json').exists():
    print('Clustering já existe:', project_dir)
else:
    cfg = BoxClusteringConfig(context_scale=1.8, min_crop_size_px=64, min_cluster_size=6, embedding_batch_size=8, dino_cpu_fallback_on_oom=True)
    result = run_box_clustering('data/roboflow/train', project_dir, cfg, overwrite=False)
    print('Objetos:', len(result['objects']))


## 9. Interface Streamlit de revisão — opcional


In [ ]:
from pyngrok import ngrok
import time
ngrok_token = userdata.get('NGROK_AUTHTOKEN')
if ngrok_token:
    ngrok.kill(); ngrok.set_auth_token(ngrok_token)
    env = os.environ.copy(); env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    streamlit_process = subprocess.Popen(['streamlit','run','app.py','--server.port','8501','--server.headless','true'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    time.sleep(3)
    print('Interface:', ngrok.connect(8501, bind_tls=True).public_url)
else:
    print('NGROK_AUTHTOKEN não configurado; etapa pulada.')


# Treinamento — etapas independentes
Antes de treinar, encerre o Streamlit para liberar VRAM.


In [ ]:
if 'streamlit_process' in globals() and streamlit_process.poll() is None: streamlit_process.terminate()
try: ngrok.kill()
except Exception: pass
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 10. Plano de treinamento e helper para executar UMA etapa

Os templates de LoRA/Full FT ficam desativados no YAML para evitar execução acidental em lote. O helper abaixo cria uma cópia de runtime e ativa somente o ID solicitado.


In [ ]:
import yaml
PLAN = Path('configs/training_experiments.yaml')
RUNTIME_PLAN = Path('workspace/training_plan_runtime.yaml')
GROUPS = ['smoke_tests','general_models','lora_specialists','full_finetune_controls']

def run_stage(stage_id, *, resume=False, dry_run=False):
    cfg = yaml.safe_load(PLAN.read_text(encoding='utf-8'))
    found = False
    for group in GROUPS:
        for key, item in (cfg.get(group, {}) or {}).items():
            if key == stage_id:
                item['enabled'] = True
                found = True
    if not found: raise KeyError(f'Etapa não encontrada: {stage_id}')
    RUNTIME_PLAN.parent.mkdir(parents=True, exist_ok=True)
    RUNTIME_PLAN.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    cmd = [sys.executable, 'scripts/train_experiments.py', '--config', str(RUNTIME_PLAN), '--only', stage_id]
    if resume: cmd.append('--resume')
    if dry_run: cmd.append('--dry-run')
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)

subprocess.run([sys.executable, 'scripts/train_experiments.py', '--group', 'all', '--list'], check=True)


## 11. Smoke Test — YOLO26n Detecção (3 épocas)
Este teste só valida ambiente, dataset e GPU. Não é o Modelo Geral final.


In [ ]:
run_stage('smoke_det_yolo26n')


## 12. MODELO GERAL — Detecção — dataset amplo
Este checkpoint é a raiz comum de todas as LoRAs de detecção e dos controles Full Fine-Tune.


In [ ]:
run_stage('det_general')
DET_GENERAL = Path('workspace/training_runs/det_general_yolo26n/weights/best.pt')
print('Modelo geral:', DET_GENERAL, 'existe=', DET_GENERAL.exists())


## 13. LoRA ESPECIALISTA — Detecção — executar UMA por vez

Cada LoRA parte de `det_general` e usa um dataset pequeno diferente. Troque apenas o número. O artefato principal será `workspace/training_runs/<id>/lora_adapter/`.


In [ ]:
DET_LORA_ID = 'det_lora_01'  # det_lora_01 ... det_lora_05
run_stage(DET_LORA_ID)


## 14. CONTROLE Full Fine-Tune — Detecção — mesmo dataset pequeno
Use o mesmo número da LoRA para comparar custo e desempenho de forma justa.


In [ ]:
DET_FULLFT_ID = DET_LORA_ID.replace('det_lora_', 'det_fullft_')
run_stage(DET_FULLFT_ID)


## 15. MODELO GERAL — Segmentação — dataset amplo
Execute quando `data/segmentation/data.yaml` estiver pronto.


In [ ]:
if Path('data/segmentation/data.yaml').exists():
    run_stage('seg_general')
else:
    print('Segmentação ainda não preparada; etapa pulada.')


## 16. LoRA ESPECIALISTA + controle Full FT — Segmentação — UMA por vez


In [ ]:
SEG_LORA_ID = 'seg_lora_01'  # seg_lora_01 ... seg_lora_05
seg_data = Path(f"data/specialists/segmentation/{SEG_LORA_ID.replace('seg_', '')}/data.yaml")
if seg_data.exists():
    run_stage(SEG_LORA_ID)
    run_stage(SEG_LORA_ID.replace('seg_lora_', 'seg_fullft_'))
else:
    print('Dataset especialista ainda não existe:', seg_data)


## 17. Comparar parâmetros e métricas de todas as etapas concluídas


In [ ]:
import pandas as pd
summary = Path('workspace/training_runs/experiment_summary.csv')
if summary.exists(): display(pd.read_csv(summary))
else: print('A tabela aparecerá após o primeiro treinamento.')


## Arquitetura de inferência que será testada depois

A LoRA não é um segundo detector autônomo. Ela é um adapter ligado ao checkpoint geral. Podemos testar: **(A)** Modelo Geral + um adapter escolhido; **(B)** Modelo Geral detecta candidatos e o ROI é reavaliado por Modelo Geral + LoRA especialista; **(C)** vários especialistas independentes e fusão das detecções. Não fazemos merge permanente das cinco LoRAs nesta etapa de treinamento.
